In [ ]:
ROOT_PATH = 'C:/Users/khoan/OneDrive/Documents/stock_data_scraper'
import os
os.chdir(ROOT_PATH)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from modules import ef_random_portfolio
from utils.constants import COMMSEC_SHARE_FEE, COMMSEC_SHARE_LARGE_FEE_PERCENTAGE
from utils.constants import COMMSEC_POCKET_FEE, COMMSEC_POCKET_LARGE_FEE_PERCENTAGE

In [ ]:
SHARE_CODES = {
    'vietnam' : ['ACB'],
    'australia' : ['TPG', 'TNE', 'SGLLV']
}
ALL_CODES = SHARE_CODES

In [ ]:
port, ret, std = ef_random_portfolio(
    codes = ALL_CODES,  
    return_method = 'mean_historical_return',
    portfolio_method = 'risk_based', 
    # portfolio_method = 'optimal',
    risk_threshold = 0.2,
    days = 365
)

In [ ]:
TOTAL_FUND = 7023.59
total_fee = 0.0
FUND_WITH_PROFIT = TOTAL_FUND * (1 + ret)
for code in port:
    investment = TOTAL_FUND * port[code]
    total_with_profit = FUND_WITH_PROFIT * port[code]
    fee = None
    for limit in (COMMSEC_SHARE_FEE if code in SHARE_CODES else COMMSEC_POCKET_FEE):
        if investment <= limit:
            fee = COMMSEC_SHARE_FEE[limit] if code in SHARE_CODES else COMMSEC_POCKET_FEE[limit]
            break
    if fee is None:
        fee = investment * (COMMSEC_SHARE_LARGE_FEE_PERCENTAGE if code in SHARE_CODES else COMMSEC_POCKET_LARGE_FEE_PERCENTAGE)
    total_fee += fee
    print(f"""[{code}] Weight: {round(port[code] * 100, 4)}%. 

    Invest: {round(investment,2)}. 
    Fee: {fee}. 
    Total: {round(investment + fee,2)}. 
    Total with profit : {round(total_with_profit, 2)}
""")
# Double the fee due to selling and buying
print(f"""Expected return: {round(ret * 100, 4)}%. 
Volatility: {round(std * 100, 4)}%.
Actual profit: {TOTAL_FUND * ret - total_fee * 2}""")